# Serialization Formats — User Guide

`StarLayerGraph.parse()`/`.serialize()` support the full set of rdflib-supported RDF formats, extended to carry RDF 1.2 content:
- `turtle12`/`longturtle12` for Turtle
- `nt12`/`nq12` for N-Triples/N-Quads
- `trig12`/`trix12` for datasets
- `rdfxml12`
- `jsonld12` — note that JSON-LD has no published RDF 1.2 spec yet; this serialization is provided for convenience, to be used within starlayer only, until the spec is finalized.

This guide is about the formats themselves; see the [Graphs guide](02-graphs.ipynb) for the triple-term/reification/direction-tagged-literal semantics being serialized, and the [datasets guide](02a-graphs-datasets.ipynb) for the dataset-capable formats applied to multiple named graphs.

## How to run this notebook

See [Getting Started](01-getting-started.ipynb) if you haven't installed StarLayer yet. This guide assumes StarLayer has been pip installed.

Run cells from top to bottom — later cells reuse variables from earlier ones.

In [1]:
from starlayer import StarLayerGraph, Namespace
from starlayergraph.compare import isomorphic

EX = Namespace("http://example.org/")

# helper output function
def test_roundtrip(graph, fmt):
    """Serialize `graph` as `fmt`, reparse it, and compare to the original.

    Prints a clean match, or - if the round trip differs - the exact
    triples present on only one side or the other.
    """
    roundtripped = StarLayerGraph()
    roundtripped.parse(data=graph.serialize(format=fmt), format=fmt)
    if isomorphic(graph, roundtripped):
        print(f"{fmt}: matches the original.")
        return
    print(f"{fmt}: differs from the original.")
    for t in sorted(set(graph) - set(roundtripped), key=str):
        print("  only in original:   ", t)
    for t in sorted(set(roundtripped) - set(graph), key=str):
        print("  only in round-trip: ", t)

## 1. `turtle12` — the reference example

**Turtle 1.2: the human-friendly, prefix-based syntax**, with all the RDF 1.2 shorthand (`<<( )>>`, `<< >>`, `{| |}`, `~`) — the format used throughout the rest of this guide set.

The document below is reused for every format shown after this section, so each one can be compared directly against the same content. It's constructed with an explicit `identifier=EX.main` (rather than leaving it as the default auto-generated one) so the dataset-capable formats later in this guide (`nq12`, `trig12`, `trix12`) have a real graph name to show — that identifier is a property of the `StarLayerGraph` object itself, set in Python, not something expressed anywhere in the Turtle text below.

In [2]:
# identifier=EX.main gives this graph a real name (rather than an
# auto-generated one) - see the markdown above for why.
g_parsed = StarLayerGraph(identifier=EX.main)
g_parsed.bind("ex", EX)
# rdflib's g.parse() is extended to handle RDF 1.2 terms for all rdflib-supported formats.
g_parsed.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

    # language-direction literals
    ex:listing_en ex:description "hello"@en--ltr .
    ex:listing_ar ex:description "مرحبا"@ar--rtl .

    # canonical reification with rdf:reifies
    ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> ;
      ex:source ex:HRSystem ;
      ex:confidence "high" .

    # reifying shorthand << s p o >>: creates an anonymous reifier
    << ex:alice ex:worksFor ex:AcmeCorp >> ex:source ex:HRSystem .

    # reifying shorthand with a named reifier: << s p o ~ id >>
    << ex:alice ex:worksFor ex:AcmeCorp ~ ex:id >> ex:source ex:HRSystem ;
      ex:reportedDate "2026-01-01"^^xsd:date .

    # anonymous inline annotation block
    ex:alice ex:likes ex:ProductABC {| ex:since "2020" ; ex:source ex:CRM |} .

    # named reifier with annotations
    ex:alice ex:mentions ex:GlobalTech ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:AuditSystem |} .

    # named reifier without annotation block
    ex:alice ex:worksWith ex:SalesTeam ~ ex:stmt2 .

    # an additional quoted triple term reused in the query examples elsewhere
    ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .
''', format='turtle12')

# rdflib's g.serialize() is extended to handle RDF 1.2 terms.
print(g_parsed.serialize(format='turtle12'))

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

ex:alice ex:likes ex:ProductABC {| ex:since "2020" ; ex:source ex:CRM |} ;
    ex:mentions ex:GlobalTech ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:AuditSystem |} ;
    ex:worksWith ex:SalesTeam ~ ex:stmt2 .

ex:claim ex:confidence "high" ;
    ex:source ex:HRSystem ;
    rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

ex:id ex:reportedDate "2026-01-01"^^xsd:date ;
    ex:source ex:HRSystem ;
    rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

ex:listing_ar ex:description "مرحبا"@ar--rtl .

ex:listing_en ex:description "hello"@en--ltr .

<< ex:alice ex:worksFor ex:AcmeCorp >> ex:source ex:HRSystem .



## 2. The other seven formats

`turtle12` above is one of eight formats `parse()`/`serialize()` support. Each subsection below covers one format: what it is, and what the reference graph looks like serialized into it.

### 2.a `longturtle12`

A **canonical, fully-expanded Turtle 1.2.** Every triple is on its own line, no shorthand blocks (`{| |}`/`~`/`<< >>` all expanded to plain `rdf:reifies` triples) — trades compactness for a form where structure is unambiguous at a glance. 

In [3]:
print(g_parsed.serialize(format="longturtle12"))

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .
ex:alice ex:likes ex:ProductABC .
ex:alice ex:mentions ex:GlobalTech .
ex:alice ex:worksWith ex:SalesTeam .
ex:claim ex:confidence "high" .
ex:claim ex:source ex:HRSystem .
ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .
ex:id ex:reportedDate "2026-01-01"^^xsd:date .
ex:id ex:source ex:HRSystem .
ex:id rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .
ex:listing_ar ex:description "مرحبا"@ar--rtl .
ex:listing_en ex:description "hello"@en--ltr .
ex:stmt1 ex:confidence "0.9" .
ex:stmt1 ex:source ex:AuditSystem .
ex:stmt1 rdf:reifies <<( ex:alice ex:mentions ex:GlobalTech )>> .
ex:stmt2 rdf:reifies <<( ex:alice ex:worksWith ex:SalesTeam )>> .
_:N4fb18f10fc3240cbad16037e1847d8b0 ex:source ex:HRSystem .
_:N4fb18f10fc3240cbad16037e1847d8b0 

In [4]:
test_roundtrip(g_parsed, "longturtle12")

longturtle12: matches the original.


### 2.b `nt12`

**N-Triples 1.2: the flattest serialization.** One full triple per line, absolute IRIs only (no prefixes), no shorthand. Easy for systems to stream line by line; not meant to be hand-written.

In [5]:
print(g_parsed.serialize(format="nt12"))

VERSION "1.2"
<http://example.org/AuditSystem> <http://example.org/reported> <<( <http://example.org/alice> <http://example.org/worksFor> <http://example.org/AcmeCorp> )>> .
<http://example.org/alice> <http://example.org/likes> <http://example.org/ProductABC> .
<http://example.org/alice> <http://example.org/mentions> <http://example.org/GlobalTech> .
<http://example.org/alice> <http://example.org/worksWith> <http://example.org/SalesTeam> .
<http://example.org/claim> <http://example.org/confidence> "high"^^<http://www.w3.org/2001/XMLSchema#string> .
<http://example.org/claim> <http://example.org/source> <http://example.org/HRSystem> .
<http://example.org/claim> <http://www.w3.org/1999/02/22-rdf-syntax-ns#reifies> <<( <http://example.org/alice> <http://example.org/worksFor> <http://example.org/AcmeCorp> )>> .
<http://example.org/id> <http://example.org/reportedDate> "2026-01-01"^^<http://www.w3.org/2001/XMLSchema#date> .
<http://example.org/id> <http://example.org/source> <http://example

In [6]:
test_roundtrip(g_parsed, "nt12")

nt12: matches the original.


### 2.c `nq12`

**N-Quads 1.2: `nt12` plus a graph name.** Same one-line-per-fact format, with a fourth term naming which graph each quad belongs to. Since it's one graph, every line below repeats the same name; a dataset with more than one named graph would show a genuinely different name per line — see the [datasets guide](02a-graphs-datasets.ipynb).

In [7]:
print(g_parsed.serialize(format="nq12"))

VERSION "1.2"
<http://example.org/AuditSystem> <http://example.org/reported> <<( <http://example.org/alice> <http://example.org/worksFor> <http://example.org/AcmeCorp> )>> <http://example.org/main> .
<http://example.org/alice> <http://example.org/likes> <http://example.org/ProductABC> <http://example.org/main> .
<http://example.org/alice> <http://example.org/mentions> <http://example.org/GlobalTech> <http://example.org/main> .
<http://example.org/alice> <http://example.org/worksWith> <http://example.org/SalesTeam> <http://example.org/main> .
<http://example.org/claim> <http://example.org/confidence> "high"^^<http://www.w3.org/2001/XMLSchema#string> <http://example.org/main> .
<http://example.org/claim> <http://example.org/source> <http://example.org/HRSystem> <http://example.org/main> .
<http://example.org/claim> <http://www.w3.org/1999/02/22-rdf-syntax-ns#reifies> <<( <http://example.org/alice> <http://example.org/worksFor> <http://example.org/AcmeCorp> )>> <http://example.org/main> .

In [8]:
test_roundtrip(g_parsed, "nq12")

nq12: matches the original.


### 2.d `trig12`

**TriG 1.2: Turtle plus `GRAPH` blocks.** Turtle's own shorthand syntax, extended so a single document can hold multiple named graphs inside `GRAPH <name> { ... }` blocks. `g_parsed` is one graph, so there's only one block here, named `ex:main`. See the [datasets guide](02a-graphs-datasets.ipynb) for multiple `GRAPH` blocks in one document.

In [9]:
print(g_parsed.serialize(format="trig12"))

@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

GRAPH <http://example.org/main> {
    @version "1.2" .

    ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

    ex:alice ex:likes ex:ProductABC {| ex:since "2020" ; ex:source ex:CRM |} ;
        ex:mentions ex:GlobalTech ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:AuditSystem |} ;
        ex:worksWith ex:SalesTeam ~ ex:stmt2 .

    ex:claim ex:confidence "high" ;
        ex:source ex:HRSystem ;
        rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

    ex:id ex:reportedDate "2026-01-01"^^xsd:date ;
        ex:source ex:HRSystem ;
        rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

    ex:listing_ar ex:description "مرحبا"@ar--rtl .

    ex:listing_en ex:description "hello"@en--ltr .

    << ex:alice ex:worksFor ex:AcmeCorp >> ex:source ex:HRSystem .
}



In [10]:
test_roundtrip(g_parsed, "trig12")

trig12: matches the original.


### 2.e `trix12`

**TriX 1.2: an XML serialization, one `<triple>` element per triple** (nested for a triple term), grouped in a `<graph>` block that names itself via a `<uri>` child — `ex:main` here. TriX was never a W3C spec — this format follows Apache Jena's convention, so it round-trips through live Fuseki's own TriX support.

In [11]:
print(g_parsed.serialize(format="trix12"))

<?xml version="1.0" encoding="UTF-8"?>
<trix xmlns="http://www.w3.org/2004/03/trix/trix-1/">
  <graph>
    <uri>http://example.org/main</uri>
    <triple>
      <id>N4fb18f10fc3240cbad16037e1847d8b0</id>
      <uri>http://example.org/source</uri>
      <uri>http://example.org/HRSystem</uri>
    </triple>
    <triple>
      <id>N4fb18f10fc3240cbad16037e1847d8b0</id>
      <uri>http://www.w3.org/1999/02/22-rdf-syntax-ns#reifies</uri>
      <triple>
        <uri>http://example.org/alice</uri>
        <uri>http://example.org/worksFor</uri>
        <uri>http://example.org/AcmeCorp</uri>
      </triple>
    </triple>
    <triple>
      <id>Na498e81936ff44d7a4ae7bb7f01a16df</id>
      <uri>http://example.org/since</uri>
      <typedLiteral datatype="http://www.w3.org/2001/XMLSchema#string">2020</typedLiteral>
    </triple>
    <triple>
      <id>Na498e81936ff44d7a4ae7bb7f01a16df</id>
      <uri>http://example.org/source</uri>
      <uri>http://example.org/CRM</uri>
    </triple>
    <triple>


In [12]:
test_roundtrip(g_parsed, "trix12")

trix12: matches the original.


### 2.f `rdfxml12`

**RDF/XML 1.2: the verbose, W3C-original XML serialization.** `rdf:Description` groups triples by subject; a triple term is carried via `rdf:parseType="Triple"`.

In [13]:
print(g_parsed.serialize(format="rdfxml12"))

<?xml version="1.0" encoding="UTF-8"?>
<rdf:RDF xmlns:ex="http://example.org/" xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#">
  <rdf:Description rdf:nodeID="N4fb18f10fc3240cbad16037e1847d8b0">
    <ex:source rdf:resource="http://example.org/HRSystem" />
    <rdf:reifies rdf:parseType="Triple">
      <rdf:Description rdf:about="http://example.org/alice">
        <ex:worksFor rdf:resource="http://example.org/AcmeCorp" />
      </rdf:Description>
    </rdf:reifies>
  </rdf:Description>
  <rdf:Description rdf:nodeID="Na498e81936ff44d7a4ae7bb7f01a16df">
    <ex:since>2020</ex:since>
    <ex:source rdf:resource="http://example.org/CRM" />
    <rdf:reifies rdf:parseType="Triple">
      <rdf:Description rdf:about="http://example.org/alice">
        <ex:likes rdf:resource="http://example.org/ProductABC" />
      </rdf:Description>
    </rdf:reifies>
  </rdf:Description>
  <rdf:Description rdf:about="http://example.org/AuditSystem">
    <ex:reported rdf:parseType="Triple">
      <rdf:D

In [14]:
test_roundtrip(g_parsed, "rdfxml12")

rdfxml12: matches the original.


### 2.g `jsonld12`

**JSON-LD, extended with a StarLayerGraph invented triple-term convention.** JSON-LD has no published RDF 1.2 companion spec yet, so `jsonld12` carries triple terms via a starlayergraph-invented `rdf:TripleTerm` node shape (visible below as the `tt:HASH`-keyed objects) — a convenience for working within StarLayer, not a spec-conformant interchange format to rely on elsewhere.

In [15]:
print(g_parsed.serialize(format="jsonld12"))

{
  "@context": {
    "tt": "https://github.com/hidden-graph/starlayergraph/ns/tt#",
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "ex": "http://example.org/"
  },
  "@graph": [
    {
      "@id": "tt:045873550c726b7b",
      "@type": [
        "rdf:TripleTerm"
      ],
      "rdf:subject": [
        {
          "@id": "ex:alice"
        }
      ],
      "rdf:predicate": [
        {
          "@id": "ex:likes"
        }
      ],
      "rdf:object": [
        {
          "@id": "ex:ProductABC"
        }
      ]
    },
    {
      "@id": "tt:072cd76e08acc3b4",
      "@type": [
        "rdf:TripleTerm"
      ],
      "rdf:subject": [
        {
          "@id": "ex:alice"
        }
      ],
      "rdf:predicate": [
        {
          "@id": "ex:mentions"
        }
      ],
      "rdf:object": [
        {
          "@id": "ex:GlobalTech"
        }
      ]
    },
    {
      "@id": "tt:a3ca2701f68d5d77",
      "@type": [
        "rdf:TripleTerm"
      ],
      "rdf:subject":

In [16]:
test_roundtrip(g_parsed, "jsonld12")

jsonld12: matches the original.


## Further Reading

1. **[Getting Started](01-getting-started.ipynb)**
2. **[Graphs](02-graphs.ipynb)** — `TripleTerm`/`DirLangString` semantics, Turtle 1.2 reification syntax.
   - 2.a **[Working with datasets](02a-graphs-datasets.ipynb)** — `StarLayerDataset`, multiple named graphs in one store; the dataset-capable formats (`trig12`, `trix12`, `nq12`) applied to multiple named graphs.
5. **Other**
   - 5.a **Serialization formats** — this guide.